## Загрузка датасета

In [86]:
import pandas as pd
import numpy as np
import re

In [87]:
df = pd.read_csv('data/hakaton.csv', sep=';')
print(f"Исходный: {len(df)} записей\n")

Исходный: 25628 записей



In [88]:
df['result'] = df['result'].str.strip().str.upper()

cols_content = [c for c in df.columns if c not in ['our_number', 'name_naprav', 'name_area']]
before = len(df)
df = df.drop_duplicates(subset=cols_content, keep='first')
print(f"[1a] Полные дубликаты: -{before - len(df)} → {len(df)}")

df = df.drop(columns=['our_number', 'name_naprav', 'name_area'])

[1a] Полные дубликаты: -183 → 25445


## Нормализация

In [89]:
for col in ['bdate', 'test_date', 'guard_bdate']:
    df[col] = pd.to_datetime(df[col])

In [90]:
df['id_doc'] = df['id_doc'].replace('-', np.nan)

In [91]:
def normalize_id(v):
    if pd.isna(v):
        return v
    cleaned = re.sub(r'[№N\-\s]', '', str(v).strip())
    if cleaned == '' or not any(c.isdigit() for c in cleaned):
        return np.nan
    return cleaned

df['id_doc'] = df['id_doc'].apply(normalize_id)
df['guard_id_doc'] = df['guard_id_doc'].apply(normalize_id)
print(f"[2b] id_doc нормализован. NaN: {df['id_doc'].isna().sum()}")

[2b] id_doc нормализован. NaN: 49


In [92]:
def normalize_variant(v):
    if pd.isna(v):
        return v
    v = str(v).strip()
    try:
        int(v)
        return v
    except ValueError:
        pass
    m = re.match(r'^УЧ\s+(\d+)', v)
    if m: return m.group(1)
    m = re.match(r'^7[24]7[._/](\d{5,7})$', v)
    if m: return m.group(1)
    m = re.match(r'^727\.(\d+)\.(\d+)$', v)
    if m: return m.group(1) + m.group(2)
    m = re.match(r'^№\s*(\d+)', v)
    if m: return m.group(1)
    m = re.match(r'^0\s+(\d+)$', v)
    if m: return '0' + m.group(1)
    m = re.match(r'^`(\d+)$', v)
    if m: return m.group(1)
    m = re.match(r'^(\d{3,})\.(\d+)$', v)
    if m: return m.group(1) + m.group(2)
    return np.nan

df['variant'] = df['variant'].apply(normalize_variant)
print(f"[2c] variant нормализован. NaN: {df['variant'].isna().sum()}")

[2c] variant нормализован. NaN: 25


In [93]:
df.loc[df['class'] == '2-5', 'class'] = '3'
print("[2d] class='2-5' → '3'")

[2d] class='2-5' → '3'


## Удаляем дубли по ключу

In [94]:
mask_valid = df['id_doc'].notna()
df_v = df[mask_valid].copy()
df_inv = df[~mask_valid].copy()
before = len(df_v)
df_v = df_v.sort_values(['id_doc', 'test_date', 'variant', 'class'])
df_v = df_v.drop_duplicates(subset=['id_doc', 'test_date', 'variant', 'class'], keep='last')
print(f"[1b] Дубликаты по ключу: -{before - len(df_v)}")
df = pd.concat([df_v, df_inv], ignore_index=True)
print(f"[1b] Итого: {len(df)}")

[1b] Дубликаты по ключу: -416
[1b] Итого: 25029


## ТОЧЕЧНЫЕ ФИКСЫ АНОМАЛИЙ

In [95]:
before = len(df)

# --- 3a. OGRN плохой длины → удаляем ---
mask_ogrn = df['ogrn_naprav'].astype(str).str.len() != 13
print(f"[3a] ogrn_naprav кривой: удалено {mask_ogrn.sum()}")
df = df[~mask_ogrn]

[3a] ogrn_naprav кривой: удалено 1


In [96]:
mask_var = df['variant'].isna()
print(f"[3b] variant невалидный: удалено {mask_var.sum()}")
df = df[~mask_var]

[3b] variant невалидный: удалено 22


In [97]:
df.loc[(df['id_doc'] == '13653549') & (df['bdate'] == pd.Timestamp('2025-09-18')), 'bdate'] = pd.Timestamp('2012-09-18')

In [98]:
df.loc[(df['id_doc'] == '0384771') & (df['bdate'] == pd.Timestamp('2025-09-14')), 'bdate'] = pd.Timestamp('2018-09-14')

# СОРОКИНА МАРИЯ (id=18151280): bdate 2025-05-16 → 2015-05-16
df.loc[(df['id_doc'] == '18151280') & (df['bdate'] == pd.Timestamp('2025-05-16')), 'bdate'] = pd.Timestamp('2015-05-16')

# КОМАРДИНА ИРИНА (id=405348032): bdate 2025-08-13 → 2011-08-13 (оставляем, близнецы)
df.loc[(df['id_doc'] == '405348032') & (df['bdate'] == pd.Timestamp('2025-08-13')), 'bdate'] = pd.Timestamp('2011-08-13')

print("[3c] Исправлены bdate: ЛУКОНИН→2012, ЗАХАРОВ→2018, СОРОКИНА→2015, КОМАРДИНА→2011")

# --- 3d. guard_same_age: bdate скопирован с родителя → восстанавливаем по классу ---
# КУЗИНА ВИКТОРИЯ (id=0535794): class=9 → ожид. ~2011
df.loc[(df['id_doc'] == '0535794') & (df['bdate'] == pd.Timestamp('2006-01-23')), 'bdate'] = pd.Timestamp('2011-01-23')

# ТОКНЕШЕВ ОЛЕГ (id=1306476): class=10 → ожид. ~2009
df.loc[(df['id_doc'] == '1306476') & (df['bdate'] == pd.Timestamp('1996-10-04')), 'bdate'] = pd.Timestamp('2009-10-04')

# НЕФЕДОВ ДМИТРИЙ (id=576809): class=1 → ожид. ~2018
df.loc[(df['id_doc'] == '576809') & (df['bdate'] == pd.Timestamp('1992-02-27')), 'bdate'] = pd.Timestamp('2018-02-27')

print("[3d] guard_same_age: bdate восстановлен по классу (КУЗИНА→2011, ТОКНЕШЕВ→2009, НЕФЕДОВ→2018)")

# --- 3e. child_too_old (без guard_same_age) → удаляем ---
df['child_age'] = (df['test_date'] - df['bdate']).dt.days / 365.25
mask_old = (df['child_age'] > 20)
print(f"[3e] child_too_old: удалено {mask_old.sum()}")
df = df[~mask_old]

# --- 3f. child_too_young: МИШИН → удаляем, БАБАЕВУ оставляем ---
mask_mishin = (df['id_doc'] == '15238017') & (df['bdate'].dt.year == 2023)
print(f"[3f] МИШИН РОМАН (нет данных для восстановления): удалено {mask_mishin.sum()}")
df = df[~mask_mishin]

# БАБАЕВА и КОМАРДИНА — оставлены
df = df.drop(columns=['child_age'])

df = df.reset_index(drop=True)
print(f"\nПосле точечных фиксов: {len(df)}")

[3c] Исправлены bdate: ЛУКОНИН→2012, ЗАХАРОВ→2018, СОРОКИНА→2015, КОМАРДИНА→2011
[3d] guard_same_age: bdate восстановлен по классу (КУЗИНА→2011, ТОКНЕШЕВ→2009, НЕФЕДОВ→2018)
[3e] child_too_old: удалено 3
[3f] МИШИН РОМАН (нет данных для восстановления): удалено 1

После точечных фиксов: 25002


In [99]:
##before = len(df)
##df = df[df['id_doc'].notna()].reset_index(drop=True)
##print(f"[4] id_doc NaN удалено: {before - len(df)} → {len(df)}")

In [100]:
# Создаём person_key из ФИО + bdate
df['person_key'] = (
    df['last_name'].str.strip() + '|' +
    df['first_name'].str.strip() + '|' +
    df['middle_name'].fillna('').str.strip() + '|' +
    df['bdate'].dt.strftime('%Y-%m-%d')
)


In [101]:
# Для каждого person_key находим «основной» id_doc (самый частый)
person_id_map = df.groupby('person_key')['id_doc'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0])

# Присваиваем единый person_id
df['person_id'] = df['person_key'].map(person_id_map)

n_persons = df['person_key'].nunique()
n_ids = df['id_doc'].nunique()
n_unified = df['person_id'].nunique()
print(f"  Уникальных person_key (ФИО+ДР): {n_persons}")
print(f"  Уникальных id_doc (до объединения): {n_ids}")
print(f"  Уникальных person_id (после): {n_unified}")

# Сколько людей имели разные id_doc, теперь объединены
multi_id = df.groupby('person_key')['id_doc'].nunique()
merged = multi_id[multi_id > 1]
print(f"  Людей с >1 id_doc (объединены): {len(merged)}")

  Уникальных person_key (ФИО+ДР): 21153
  Уникальных id_doc (до объединения): 21193
  Уникальных person_id (после): 20509
  Людей с >1 id_doc (объединены): 692


In [102]:
print(f"\n{'='*60}")
print("ШАГ 6: ВЫЯВЛЕНИЕ АНОМАЛИЙ")
print(f"{'='*60}")

# --- 6a. Нарушения частоты (<90 дней) по person_id ---
df_s = df.sort_values(['person_id', 'test_date']).copy()
df_s['prev_test'] = df_s.groupby('person_id')['test_date'].shift(1)
df_s['days_gap'] = (df_s['test_date'] - df_s['prev_test']).dt.days

freq_mask = df_s['days_gap'].notna() & (df_s['days_gap'] < 90)
freq_idx = df_s[freq_mask].index
df['freq_violation'] = df.index.isin(freq_idx)

same_day_idx = df_s[df_s['days_gap'] == 0].index
df['same_day_test'] = df.index.isin(same_day_idx)

print(f"  Нарушения частоты (<90 дней): {df['freq_violation'].sum()} записей, {df_s[freq_mask]['person_id'].nunique()} детей")
print(f"  Из них в один день: {df['same_day_test'].sum()}")

# --- 6b. multi_class: разница классов > 1 ---
df['class_num'] = pd.to_numeric(df['class'], errors='coerce')

def check_multi_class(group):
    """Проверяем: есть ли скачки класса > 1 у ребёнка."""
    g = group.sort_values('test_date')
    classes = g['class_num'].dropna().values
    if len(classes) < 2:
        return False
    for i in range(1, len(classes)):
        if abs(classes[i] - classes[i-1]) > 1:
            return True
    return False

mc_persons = df.groupby('person_id').apply(check_multi_class)
mc_ids = mc_persons[mc_persons].index
df['multi_class_jump'] = df['person_id'].isin(mc_ids)
print(f"  Скачки класса (>1): {df['multi_class_jump'].sum()} записей, {len(mc_ids)} детей")

# --- 6c. id_doc == guard_id_doc ---
df['id_equals_guard'] = (df['id_doc'] == df['guard_id_doc'])
print(f"  id_doc == guard_id_doc: {df['id_equals_guard'].sum()}")

# --- 6d. Возраст vs класс ---
df['child_age'] = (df['test_date'] - df['bdate']).dt.days / 365.25
expected = df['class_num'] + 6
df['age_class_mismatch'] = ((df['child_age'] - expected).abs() > 3) & expected.notna()
print(f"  Несоответствие возраст-класс (>3 лет): {df['age_class_mismatch'].sum()}")

# --- 6e. guard_too_young_parent ---
df['parent_age_at_birth'] = (df['bdate'] - df['guard_bdate']).dt.days / 365.25
df['guard_too_young'] = (df['parent_age_at_birth'] > 0) & (df['parent_age_at_birth'] < 14)
print(f"  Родитель слишком молод (<14 лет): {df['guard_too_young'].sum()}")


ШАГ 6: ВЫЯВЛЕНИЕ АНОМАЛИЙ
  Нарушения частоты (<90 дней): 1082 записей, 907 детей
  Из них в один день: 182
  Скачки класса (>1): 446 записей, 159 детей
  id_doc == guard_id_doc: 324
  Несоответствие возраст-класс (>3 лет): 385
  Родитель слишком молод (<14 лет): 25


In [103]:
print(f"  Исходных записей: 25628")
print(f"  Чистых записей: {len(df)}")
print(f"  Уникальных детей (person_id): {df['person_id'].nunique()}")

anomaly_cols = ['freq_violation', 'same_day_test', 'multi_class_jump',
                'id_equals_guard', 'age_class_mismatch', 'guard_too_young']
df['has_anomaly'] = df[anomaly_cols].any(axis=1)
print(f"  Записей с аномалиями: {df['has_anomaly'].sum()} ({df['has_anomaly'].mean()*100:.1f}%)")
print(f"  Чистых записей (без аномалий): {(~df['has_anomaly']).sum()}")

  Исходных записей: 25628
  Чистых записей: 25002
  Уникальных детей (person_id): 20509
  Записей с аномалиями: 1837 (7.3%)
  Чистых записей (без аномалий): 23165


In [104]:
export_cols = [c for c in df.columns if c not in ['person_key', 'class_num', 'child_age', 'parent_age_at_birth']]
df[export_cols].to_csv('clean_final.csv', index=False)
print(f"\nСохранено: clean_final.csv ({len(df)} записей)")


Сохранено: clean_final.csv (25002 записей)
